# Dimuon mass spectrum

A minimal example: plot the invariant-mass distribution of the selected
$B_s^0 \to \mu^+\mu^-$ candidates, without any fit.

This notebook can be used in two ways:

- **interactively**: open it in Jupyter (run `snakemake --cores 16` first,
  so that `results/selected_data.root` exists);
- **through snakemake**: the `mass_plot` rule executes it headlessly and
  saves the figure to `figures/mass_plot.png`.

In [ ]:
import ROOT
%jsroot on

In [ ]:
# When executed by snakemake the `snakemake` object is injected and provides
# the input/output paths (relative to the repository root).  When the
# notebook is opened interactively, fall back to paths relative to the
# notebooks/ directory and do not write any output file.
try:
    DATA, HELPER = snakemake.input.data, snakemake.input.helper
    OUTPUT = snakemake.output[0]
except NameError:
    DATA, HELPER = "../results/selected_data.root", "../src/roofit_helper.hpp"
    OUTPUT = None

Load the selected candidates and book the mass histogram.
The bin edges are aligned so that the $B_s^0$ mass sits at a bin *center*:
the mass resolution is only $\sigma \approx 23$ MeV/$c^2$, so this keeps
the signal from being split between two bins.

In [ ]:
M_BS, M_BD = 5366.9, 5279.7  # PDG masses [MeV]
BIN_W = 40.0
LO = M_BS - 20 - 16 * BIN_W  # 4706.9, Bs at a bin center
HI = M_BS + 20 + 15 * BIN_W  # 5986.9
NBINS = round((HI - LO) / BIN_W)

df = ROOT.RDataFrame("DecayTree", DATA)
hist = df.Histo1D(("h_mass",
                   ";m(#mu^{+}#mu^{-}) [MeV/c^{2}];Candidates / (40 MeV/c^{2})",
                   NBINS, LO, HI), "B_s0_MM")
print(f"{df.Count().GetValue()} candidates in total")

Draw it in the LHCb style (from `src/roofit_helper.hpp`), with the known
$B^0$ and $B_s^0$ masses marked. Can you spot the $B_s^0 \to \mu^+\mu^-$
peak on top of the smooth background?

In [ ]:
ROOT.gInterpreter.Declare(f'#include "{HELPER}"')
ROOT.roofit_helper.lhcbStyle()

canvas = ROOT.TCanvas("canvas", "", 800, 600)
hist.SetMarkerStyle(20)
hist.SetMinimum(0)
hist.Draw("E1")

lines = []
for mass, color in [(M_BS, ROOT.kRed), (M_BD, ROOT.kBlue)]:
    line = ROOT.TLine(mass, 0, mass, hist.GetMaximum())
    line.SetLineColor(color)
    line.SetLineStyle(ROOT.kDashed)
    line.Draw()
    lines.append(line)

legend = ROOT.TLegend(0.60, 0.70, 0.92, 0.92)
legend.SetFillStyle(0)
legend.AddEntry(hist.GetValue(), "Selected candidates", "ep")
legend.AddEntry(lines[0], "m(B_{s}^{0}) PDG", "l")
legend.AddEntry(lines[1], "m(B^{0}) PDG", "l")
legend.Draw()

if OUTPUT:
    canvas.SaveAs(OUTPUT)
canvas.Draw()